In [95]:
import yfinance as yf
import numpy as np
import pandas as pd
!pip install yfinance numpy scipy plotly ipywidgets --quiet
from google.colab import output
output.enable_custom_widget_manager()

In [96]:
TICKER = "AAPL"
ticker = yf.Ticker(TICKER)
hist_1d = ticker.history(period="1d")
S = hist_1d["Close"].iloc[-1]

In [97]:
hist_60d = ticker.history(period="60d")["Close"]
returns = np.log(hist_60d / hist_60d.shift(1)).dropna()
sigma_hist = returns.std() * np.sqrt(252)
print(sigma_hist)

0.2640640132256202


In [98]:
r = 0.04306 #We used the US 10-year Yield

In [99]:
print(f"Underlying Asset : {TICKER}")
print(f"Spot Price : {S:.2f}")
print(f"Annualized Volatility : {sigma_hist*100:.1f}%")
print(f"Risk-Free rate : {r*100}%")

Underlying Asset : AAPL
Spot Price : 271.06
Annualized Volatility : 26.4%
Risk-Free rate : 4.306%


In [100]:
import plotly.graph_objects as go
hist_1y = ticker.history(period="1y")
fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=hist_1y.index,
    open=hist_1y["Open"],
    high=hist_1y["High"],
    low=hist_1y["Low"],
    close=hist_1y["Close"],
    name=TICKER
))
fig.add_hline(y=S, line_dash="dash", line_color="orange",
              annotation_text=f"Current Spot : ${S:.2f}")
fig.update_layout(
    title=f"{TICKER} — Price over a year",
    xaxis_title="Date",
    yaxis_title="Prix ($)",
    template="plotly_dark",
    height=400,
    xaxis_rangeslider_visible=False
)
fig.show()

In [101]:
hist_2y = ticker.history(period="2y")["Close"]
rets = np.log(hist_2y / hist_2y.shift(1)).dropna()

vol_20  = rets.rolling(20).std()  * np.sqrt(252) * 100
vol_30  = rets.rolling(30).std()  * np.sqrt(252) * 100
vol_60  = rets.rolling(60).std()  * np.sqrt(252) * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=vol_20.index, y=vol_20, name="Vol 20d", line=dict(width=1)))
fig.add_trace(go.Scatter(x=vol_30.index, y=vol_30, name="Vol 30d", line=dict(width=2)))
fig.add_trace(go.Scatter(x=vol_60.index, y=vol_60, name="Vol 60d", line=dict(width=1.5, dash="dash")))
fig.update_layout(
    title=f"Historical volatility — {TICKER}",
    yaxis_title="Annualized volatility (%)",
    template="plotly_dark",
    height=350
)
fig.show()

In [102]:
# Let us start by defining a function computing d1, d2 and the price of an option using BSM
from scipy.stats import norm
def black_scholes(S, K, T, r, sigma, option_type="call"):
  """
  S : spot price of the underlying asset
  K : strike price
  T : time to maturity (in years)
  r : risk-free rate
  sigma : volatility of the underlying asset
  option_type : "call" or "put"
  """

  if T <= 0:
    return max(S - K, 0) if option_type == "call" else max(K - S, 0)

  d1 = (np.log(S / K) + (r + sigma ** 2 / 2) * T) / (sigma * np.sqrt(T))
  d2 = d1 - sigma * np.sqrt(T)
  if option_type == "call":
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
  else:
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

In [103]:
# We shall now create a function to compute the most relevant Greeks
def compute_greeks(S, K, T, r, sigma, option_type="call"):
  """
  Delta: sensitivity of the option to variations of the underlying asset
  Gamma: sensitivity of the Delta to variations of the asset
  Vega: sensitivity of the option to variations of implicit volatility
  Rho: sensitivity of the option to variations of interest rates
  Theta: variations of the option price to the passing of time
  """
  d1 = (np.log(S / K) + (r + sigma ** 2 / 2) * T) / (sigma * np.sqrt(T))
  d2 = d1 - sigma * np.sqrt(T)

  if option_type=="call":
    delta = norm.cdf(d1)
    theta = -(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T)) - r*K*np.exp(-r*T)*norm.cdf(d2)
    rho = K*T*np.exp(-r*T)*norm.cdf(d2)
  else:
    delta = -norm.cdf(-d1)
    theta = -(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T)) + r*K*np.exp(-r*T)*norm.cdf(-d2)
    rho = -K*T*np.ext(-r*T)*norm.cdf(-d2)

  gamma = norm.pdf(d1)/S*sigma*np.sqrt(T)
  vega = S*np.sqrt(T)*norm.pdf(d1)
  return {"Delta": delta, "Gamma": gamma, "Vega": vega,
            "Theta": theta, "Rho": rho}

In [104]:
#We now want to use the Newton-Raphson model to find the implicit volatility of the asset

def implied_volatility(market_price, S, K, T, r, option_type="call",
                       tol=1e-6, max_iter=200):

    sigma = 0.20
    for _ in range(max_iter):
        price = black_scholes(S, K, T, r, sigma, option_type)
        vega  = compute_greeks(S, K, T, r, sigma, option_type)["Vega"] * 100
        diff  = price - market_price
        if abs(diff) < tol:
            return sigma
        if vega < 1e-10:
            break
        sigma -= diff / vega
        sigma  = max(0.001, min(sigma, 10.0))
    return None

In [105]:
#We now want to create and price an ATM call
K = round(S)
T = 90/365 #Maturity is 3 months
option_type = "call"
price_bs = black_scholes(S, K, T, r, sigma_hist, option_type)
greeks_bs = compute_greeks(S, K, T, r, sigma_hist, option_type)

if S>K*1.01:
  moneyness = "ITM"
elif S<K*0.99:
  moneyness = "OTM"
else:
  moneyness = "ATM"

print(f"Option: {option_type} on {TICKER}")
print(f"  Spot (S)      : ${S:.2f}       Strike (K) : ${K:.0f} ")
print(f"  Maturity (T)  : {T*350:.0f} days")
print(f"  Moneyness     : {moneyness}")
print(f"  Risk-free rate: {r*100:.1f}%")
print(f"  Volatility    : {sigma_hist*100:.1f}%")
print(f"  Price using BS       : ${price_bs:.4f} ")
print(f"\n  Greeks :")
for name, val in greeks_bs.items():
    unit_map = {
        "Delta": "$ / $1 on the asset",
        "Gamma": "Δdelta / $1 on the asset",
        "Vega" : "$ / +1% of vol",
        "Theta": "$ / day",
        "Rho"  : "$ / +1% on rates"
    }
    print(f"    {name:6s} = {val:+.5f}   ({unit_map[name]})")

Option: call on AAPL
  Spot (S)      : $271.06       Strike (K) : $271 
  Maturity (T)  : 86 days
  Moneyness     : ATM
  Risk-free rate: 4.3%
  Volatility    : 26.4%
  Price using BS       : $15.6020 

  Greeks :
    Delta  = +0.55892   ($ / $1 on the asset)
    Gamma  = +0.00019   (Δdelta / $1 on the asset)
    Vega   = +53.11040   ($ / +1% of vol)
    Theta  = -34.29043   ($ / day)
    Rho    = +33.50908   ($ / +1% on rates)


In [106]:
import ipywidgets as widgets
from IPython.display import display, clear_output


w_K     = widgets.FloatSlider(value=K, min=S*0.70, max=S*1.30, step=0.5,
                               description="Strike K", style={"description_width":"80px"})
w_T     = widgets.IntSlider(value=90, min=7, max=730, step=7,
                             description="Maturity (d)", style={"description_width":"80px"})
w_sigma = widgets.FloatSlider(value=sigma_hist*100, min=5, max=80, step=0.5,
                               description="Vol (%)", style={"description_width":"80px"})
w_type  = widgets.ToggleButtons(options=["call", "put"], description="Type",
                                 style={"description_width":"50px"})
w_out   = widgets.Output()

def update(_):
    K_     = w_K.value
    T_     = w_T.value / 365
    sigma_ = w_sigma.value / 100
    opt_   = w_type.value

    price_  = black_scholes(S, K_, T_, r, sigma_, opt_)
    greeks_ = compute_greeks(S, K_, T_, r, sigma_, opt_)

    with w_out:
        clear_output(wait=True)

        # Graphic price vs spot
        spots_ = np.linspace(S * 0.60, S * 1.40, 300)
        prices_= [black_scholes(s, K_, T_, r, sigma_, opt_) for s in spots_]

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=spots_, y=prices_,
                                  line=dict(color="#00d4aa", width=2.5),
                                  name="Prix BS"))
        fig.add_vline(x=S,  line_dash="dash",  line_color="white",
                      annotation_text=f"Spot {S:.1f}€")
        fig.add_vline(x=K_, line_dash="dot",   line_color="orange",
                      annotation_text=f"Strike ${K_:.1f}")
        fig.add_annotation(x=S*1.35, y=max(prices_)*0.85,
                            text=(f"<b>Price : ${price_:.3f}</b><br>"
                                  f"Δ={greeks_['Delta']:+.3f}  Γ={greeks_['Gamma']:.4f}<br>"
                                  f"ν={greeks_['Vega']:.3f}  Θ={greeks_['Theta']:.4f}"),
                            showarrow=False, bgcolor="#1a1a2e",
                            font=dict(color="white", size=12),
                            bordercolor="#00d4aa", borderwidth=1)
        fig.update_layout(
            title=f"{opt_.upper()} {TICKER} | K=${K_:.0f} | T={w_T.value}d | σ={w_sigma.value:.0f}%",
            xaxis_title="Price of the underlying in $",
            yaxis_title="Price of the option",
            template="plotly_dark", height=380,
            margin=dict(l=40, r=20, t=50, b=40)
        )
        fig.show()


for w in [w_K, w_T, w_sigma, w_type]:
    w.observe(update, names="value")


ui = widgets.VBox([
    widgets.HBox([w_type, w_K]),
    widgets.HBox([w_T, w_sigma]),
    w_out
])
display(ui)
update(None)